In [1]:
!pip install feedparser beautifulsoup4 sentence-transformers faiss-cpu transformers python-dateutil scikit-learn requests


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 832.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 22.6 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=e7623ca6612557eedd9d41564b2492c76091341da3ce02a438a4a02457f7b740
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [2]:
import re
import requests
import feedparser
from bs4 import BeautifulSoup
from datetime import timezone
from dateutil import parser as dateparser

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer
import faiss

from transformers import GPT2LMHeadModel, GPT2Tokenizer


In [3]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2-large")
gpt2 = GPT2LMHeadModel.from_pretrained("openai-community/gpt2-large")

tokenizer.pad_token = tokenizer.eos_token

print("Models loaded.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Models loaded.


In [4]:
def clean_text(raw_html: str) -> str:
    """
    Clean RSS summary / HTML content:
    - remove tags
    - drop images (and their alt text/captions)
    - collapse whitespace
    """
    if not raw_html:
        return ""

    soup = BeautifulSoup(raw_html, "html.parser")

    for img in soup.find_all("img"):
        img.decompose()

    text = soup.get_text(separator=" ")

    text = re.sub(r"\s+", " ", text).strip()
    return text


In [5]:
def safe_feedparse(url, timeout=5):
    """Fetch RSS safely with timeout and error handling."""
    try:
        resp = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        return feedparser.parse(resp.text)
    except Exception as e:
        print("Failed to fetch:", url, "→", e)
        return feedparser.parse("")

def fetch_articles_from_rss(source_name, feed_url, max_articles=20):
    feed = safe_feedparse(feed_url)
    print(source_name, "| entries in feed:", len(feed.entries))

    docs = []
    for entry in feed.entries[:max_articles]:
        title = entry.get("title", "").strip()
        summary = entry.get("summary", "").strip()
        url = entry.get("link", "")

        clean_summary = clean_text(summary)
        text = clean_summary or title
        if not text:
            continue

        docs.append({
            "source": source_name,
            "title": title,
            "text": text,
            "url": url,
            "published": entry.get("published", "")
        })

    return docs


In [6]:
NEWS_SOURCES = {
    "Global News BC": "https://globalnews.ca/bc/feed/",
    "CBC BC": "https://www.cbc.ca/webfeed/rss/rss-canada-britishcolumbia",
    "The Province": "https://theprovince.com/feed",

    "CBC Canada": "https://www.cbc.ca/webfeed/rss/rss-canada",


    "CNN World": "http://rss.cnn.com/rss/edition_world.rss",
    "BBC World": "http://feeds.bbci.co.uk/news/world/rss.xml",
}


In [54]:
all_documents = []

for name, url in NEWS_SOURCES.items():
    docs = fetch_articles_from_rss(name, url, max_articles=50)
    print(" → Retrieved:", len(docs))
    all_documents.extend(docs)

print("Total articles:", len(all_documents))


Global News BC | entries in feed: 10
 → Retrieved: 10
CBC BC | entries in feed: 20
 → Retrieved: 20
The Province | entries in feed: 10
 → Retrieved: 10
CBC Canada | entries in feed: 20
 → Retrieved: 20
CNN World | entries in feed: 29
 → Retrieved: 29
BBC World | entries in feed: 37
 → Retrieved: 37
Total articles: 126


In [55]:
len(all_documents), all_documents[0]


(126,
 {'source': 'Global News BC',
  'title': 'B.C. pulp mill to close, leaving 350 employees out of work',
  'text': 'Domtar, which owns the mill in Crofton, says continued poor pricing for pulp and a lack of access to affordable fibre led to the plant closure.',
  'url': 'https://globalnews.ca/news/11557907/bc-pulp-mill-crofton-close-forestry/',
  'published': 'Wed, 03 Dec 2025 19:42:42 +0000'})

In [56]:
corpus_texts = [
    doc["title"] + "\n\n" + doc["text"]
    for doc in all_documents
]

corpus_embeddings = embed_model.encode(corpus_texts, convert_to_numpy=True).astype("float32")

embedding_dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(corpus_embeddings)

print("FAISS index ready. Documents indexed:", len(corpus_texts))


FAISS index ready. Documents indexed: 126


In [58]:
import re
from sklearn.metrics.pairwise import cosine_similarity

REGION_KEYWORDS = {
    "vancouver": [
        "vancouver", "downtown vancouver", "granville", "kitsilano",
        "mount pleasant", "gastown", "yvr"
    ],
    "bc": [
        "b.c.", "british columbia", "vancouver", "surrey", "burnaby",
        "richmond", "delta", "coquitlam", "langley", "abbotsford",
        "nanaimo", "kelowna", "kamloops", "prince george", "cranbrook",
    ],
}


In [59]:
def is_bc_news(doc):
    title = doc.get("title", "").lower()
    summary = doc.get("text", "").lower()

    bc_keywords = [
        "british columbia", "b.c.", " bc ",
        "vancouver", "surrey", "burnaby", "richmond",
        "coquitlam", "delta", "new westminster",
        "north vancouver", "west vancouver",
        "langley", "abbotsford", "chilliwack",
        "victoria", "nanaimo", "kelowna", "kamloops",
        "lytton", "cranbrook",
    ]

    return any(k in title or k in summary for k in bc_keywords)


bc_documents = [d for d in all_documents if is_bc_news(d)]

print("Raw documents loaded:", len(all_documents))
print("BC / Vancouver filtered documents:", len(bc_documents))



Raw documents loaded: 126
BC / Vancouver filtered documents: 32


In [60]:
from sklearn.neighbors import NearestNeighbors


bc_texts_for_index = [
    (d.get("title", "") + " " + d.get("text", "")).strip()
    for d in bc_documents
]

if bc_texts_for_index:
    bc_embeddings = embed_model.encode(bc_texts_for_index, convert_to_numpy=True)

    nn_model = NearestNeighbors(metric="cosine")
    nn_model.fit(bc_embeddings)

    print("Embeddings shape:", bc_embeddings.shape)
else:
    bc_embeddings = None
    nn_model = None
    print("No BC docs available for indexing.")


Embeddings shape: (32, 384)


In [37]:
print("Raw documents loaded:", len(all_documents))

bc_documents = [d for d in all_documents if is_bc_news(d)]

print("BC / Vancouver filtered documents:", len(bc_documents))


Raw documents loaded: 126
BC / Vancouver filtered documents: 31


In [12]:
def doc_matches_region(doc, region: str) -> bool:
    """
    Return True if this document looks related to the region.
    Uses simple keyword search over title + text.
    """
    if not region:
        return True

    region = region.lower()
    keywords = REGION_KEYWORDS.get(region, [region])

    haystack = (doc.get("title", "") + " " + doc.get("text", "")).lower()
    return any(kw in haystack for kw in keywords)


In [101]:
def retrieve_docs(query, k=3):
    """
    Retrieve top-k BC/Vancouver documents for a query using cosine similarity.
    Uses bc_documents + bc_embeddings + nn_model.
    """
    if not bc_documents or nn_model is None or bc_embeddings is None:
        return []

    q_emb = embed_model.encode([query], convert_to_numpy=True)

    k_eff = min(k, len(bc_documents))

    distances, indices = nn_model.kneighbors(q_emb, n_neighbors=k_eff)

    docs = []
    for dist, idx in zip(distances[0], indices[0]):
        if 0 <= idx < len(bc_documents):
            doc = bc_documents[idx].copy()
            doc["distance"] = float(dist)
            docs.append(doc)

    return docs

In [62]:
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def extractive_summary(text, n_sentences=2):
    """
    Extractive summary using sentence embeddings.
    Cleans HTML, avoids tiny fragments, keeps sentence order.
    """
    if not text:
        return ""

    text_clean = re.sub(r"<[^>]+>", " ", text)
    text_clean = re.sub(r'\s+', ' ', text_clean).strip()

    sentences = re.split(r'(?<=[.!?])\s+', text_clean)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 5]

    if not sentences:
        return ""

    if len(sentences) <= n_sentences:
        return " ".join(sentences)

    sent_embs = embed_model.encode(sentences, convert_to_numpy=True)
    doc_emb = embed_model.encode([text_clean], convert_to_numpy=True)[0]

    sims = cosine_similarity(sent_embs, doc_emb.reshape(1, -1)).flatten()

    top_idx = np.argsort(sims)[-n_sentences:]
    top_idx = sorted(top_idx)

    return " ".join(sentences[i] for i in top_idx)

In [16]:
def summarize_doc(doc, n_sentences=2):
    source = doc["source"]
    title = doc["title"]
    text = doc["text"]
    url = doc.get("url", "")

    short = extractive_summary(text, n_sentences=n_sentences)
    if not short:
        return "No content to summarize."

    line = f"{short} (Source: {source} — {title})"
    if url:
        line += f"\nLink: {url}"
    return line


In [42]:
sample = bc_documents[0]
print(summarize_doc(sample))


Domtar, which owns the mill in Crofton, says continued poor pricing for pulp and a lack of access to affordable fibre led to the plant closure. (Source: Global News BC — B.C. pulp mill to close, leaving 350 employees out of work)
Link: https://globalnews.ca/news/11557907/bc-pulp-mill-crofton-close-forestry/


In [43]:
def get_published_datetime(doc):
    pub = doc.get("published", "")
    if not pub:
        return None
    try:
        dt = dateparser.parse(pub)
        if dt is None:
            return None
        if dt.tzinfo is not None:
            dt = dt.astimezone(timezone.utc).replace(tzinfo=None)
        return dt
    except Exception:
        return None

def summarize_latest_news(n=5):
    dated_docs = []
    for d in bc_documents:
        dt = get_published_datetime(d)
        if dt is not None:
            dated_docs.append((dt, d))

    if not dated_docs:
        return "No recent articles available."

    dated_docs.sort(key=lambda x: x[0], reverse=True)
    top_docs = [d for _, d in dated_docs[:n]]

    summaries = []
    for doc in top_docs:
        summaries.append(summarize_doc(doc, n_sentences=2))

    if not summaries:
        return "No recent articles available."

    return "\n\n".join(summaries)


In [19]:
print(summarize_latest_news(n=5))


As the lowest-ranked side in the expanded 24-team tournament, the Canadian men were always going to get a tough draw. (Source: The Province — Canada men draw Argentina, Fiji and Spain at 2027 Rugby World Cup in Australia)
Link: https://theprovince.com/sports/rugby/canada-men-draw-argentina-fiji-spain-2027-rugby-world-cup-australia

Domtar, which owns the mill in Crofton, says continued poor pricing for pulp and a lack of access to affordable fibre led to the plant closure. (Source: Global News BC — B.C. pulp mill to close, leaving 350 employees out of work)
Link: https://globalnews.ca/news/11557907/bc-pulp-mill-crofton-close-forestry/

Ukrainian Foreign Minister Andrii Sybiha‎ spoke after high-stakes talks on ending the war failed to produce tangible results. (Source: BBC World — Stop wasting the world's time, Ukraine tells Putin after US talks in Moscow)
Link: https://www.bbc.com/news/articles/c7731n5z6d3o?at_medium=RSS&at_campaign=rss

Aisha Estey along with six other members of the 

/usr/local/lib/python3.12/dist-packages/dateutil/parser/_parser.py:1207: UnknownTimezoneWarning: tzname EST identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "
/usr/local/lib/python3.12/dist-packages/dateutil/parser/_parser.py:1207: UnknownTimezoneWarning: tzname EDT identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "


In [20]:
def search_articles(query, k=5):
    retrieved = retrieve_docs(query, k=k)
    if len(retrieved) == 0:
        print("No relevant news found for this query.")
        return []

    for idx, doc in enumerate(retrieved):
        print(f"[{idx}] {doc['title']} ({doc['source']})")
        if doc.get("published"):
            print("     Published:", doc["published"])
        if doc.get("url"):
            print("     Link:", doc["url"])
        print()
    return retrieved


In [21]:
candidates = search_articles("pipeline northern BC", k=5)


[0] B.C. Hydro taking family to court over blocked work on major transmission line (CBC BC)
     Published: Wed, 03 Dec 2025 15:25:28 EST
     Link: https://www.cbc.ca/news/canada/british-columbia/bc-hydro-north-coast-transmission-line-court-case-9.7001827?cmp=rss

[1] WATCH: Global News Hour at 6 BC: Dec. 2 (Global News BC)
     Published: Wed, 03 Dec 2025 00:00:41 +0000
     Link: https://globalnews.ca/news/1149299/watch-news-hour/

[2] Class-action lawsuit certified against CN and CP railways over Lytton, B.C., fire (CBC BC)
     Published: Tue, 02 Dec 2025 18:06:37 EST
     Link: https://www.cbc.ca/news/canada/british-columbia/lytton-bc-fire-class-action-lawsuit-cn-cp-9.7000779?cmp=rss

[3] Class-action lawsuit certified against CN and CP railways over Lytton, B.C., fire (CBC Canada)
     Published: Tue, 02 Dec 2025 18:06:37 EST
     Link: https://www.cbc.ca/news/canada/british-columbia/lytton-bc-fire-class-action-lawsuit-cn-cp-9.7000779?cmp=rss

[4] Not only is Lake Powell's water

In [22]:
def tidy_summary(text: str) -> str:
    """
    Light cleanup for readability.
    - Fix common run-ons like 'says The' → 'says. The'
    - Collapse weird extra spaces
    """
    text = re.sub(r"\b(says|said)\s+([A-Z])", r"\1. \2", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text


In [23]:
def rewrite_with_gpt2_xl(extractive_text, max_new_tokens=80):
    """
    Use GPT-2 XL to rewrite an extractive summary into a short, fluent paragraph.
    We tell it explicitly NOT to add new information.
    """
    text = (extractive_text or "").strip()
    if len(text) < 10:
        return extractive_text

    prompt = (
        "You are a news summarization assistant.\n"
        "Rewrite the following text as a concise local news summary (2–3 sentences).\n"
        "Keep all facts the same. Do NOT add any new information.\n\n"
        "TEXT:\n"
        f"{text}\n\n"
        "SUMMARY:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True,
    )

    output_ids = gpt2.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    summary = tokenizer.decode(generated, skip_special_tokens=True).strip()

    if len(summary) < 20:
        return extractive_text

    return summary


In [24]:
import re

def fused_summary_hybrid(docs, n_sentences=4, use_generative=True):
    """
    1) Extractive summarization over retrieved docs (factual).
    2) Optional GPT-2 XL rewrite to improve fluency.
    3) Append source list.
    """

    combined_text = " ".join(d["text"] for d in docs)

    extracted = extractive_summary(combined_text, n_sentences=n_sentences)
    extracted = re.sub(r'\s+', ' ', extracted).strip()

    if use_generative:
        rewritten = rewrite_with_gpt2_xl(extracted, max_new_tokens=96)
    else:
        rewritten = extracted

    sources_block = "\n".join([
        f"- {d['source']} — {d['title']} ({d.get('url','')})"
        for d in docs
    ])

    final = f"{rewritten}\n\nSources:\n{sources_block}"
    return final


In [125]:
def fused_summary(docs, n_sentences=4):
    """
    Combine multiple articles into ONE clean extractive summary.
    """
    if not docs:
        return "No relevant news found for this query."

    combined_text = " ".join(d["text"] for d in docs if d.get("text"))
    if not combined_text.strip():
        return "No relevant news content available."

    extracted = extractive_summary(combined_text, n_sentences=n_sentences)
    extracted = re.sub(r"\s+", " ", extracted).strip()

    sources_block = "\n".join([
        f"- {d['source']} — {d['title']} ({d.get('url', '')})"
        for d in docs
    ])

    final = f"{extracted}\n\nSources:\n{sources_block}"
    return final



In [70]:
import re

def find_keyword_matches(query, docs, max_docs=10):
    """
    Return docs that contain at least one non-trivial word from the query
    in their title or text.
    """
    q = query.lower()
    tokens = [w for w in re.split(r"\W+", q) if len(w) > 3]

    if not tokens:
        return []

    matches = []
    for d in docs:
        title = d.get("title", "").lower()
        text = d.get("text", "").lower()
        for t in tokens:
            if t in title or t in text:
                matches.append(d)
                break


    return matches[:max_docs]


In [119]:
import re


STOPWORDS = {
    "what", "happened", "story", "about", "the", "a", "an",
    "in", "on", "of", "for", "to", "with", "recent", "latest",
    "summarize", "summary", "tell", "me", "news", "case",
    "issue", "situation", "update", "updates"
}

def get_strong_tokens(query: str):
    """
    Extract 'strong' content words from the user query:
    - lowercase
    - at least 4 characters
    - not in a small stopword list
    """
    tokens = re.findall(r"[a-zA-Z]+", query.lower())
    strong = [
        t for t in tokens
        if len(t) >= 4 and t not in STOPWORDS
    ]
    return strong



def filter_docs_by_tokens(docs, strong_tokens, min_matches: int = 2):
    """
    Keep only docs that contain at least `min_matches` of the strong tokens
    in their title+text. If strong_tokens is empty, just return docs unchanged.
    """
    if not strong_tokens:
        return docs

    filtered = []
    for d in docs:
        haystack = (d.get("title", "") + " " + d.get("text", "")).lower()
        count = sum(1 for t in strong_tokens if t in haystack)
        if count >= min_matches:
            filtered.append(d)

    return filtered



In [120]:
print(get_strong_tokens("What happened with the pistachio salmonella restrictions?"))

['pistachio', 'salmonella', 'restrictions']


In [121]:
def title_keyword_search(strong_tokens, max_docs=3):
    """
    Fallback: search ALL documents (BC + others) by keyword overlap
    in TITLE + TEXT. Returns docs sorted by how many strong tokens they match.
    """
    if not strong_tokens:
        return []

    candidates = []
    for d in all_documents:
        haystack = (d.get("title", "") + " " + d.get("text", "")).lower()
        score = sum(1 for t in strong_tokens if t in haystack)
        if score > 0:
            candidates.append((score, d))

    candidates.sort(key=lambda x: x[0], reverse=True)
    return [d for score, d in candidates[:max_docs]]



In [122]:
tokens = get_strong_tokens("What happened with the pistachio salmonella restrictions?")
hits = title_keyword_search(tokens, max_docs=5)

for d in hits:
    print(d["source"], "—", d["title"])


Global News BC — Canada puts new restrictions on pistachios from Iran amid salmonella outbreak
CBC Canada — What to know about the imported pistachios linked to lasting salmonella outbreak


In [71]:
def safe_retrieve(query, k=3):
    """
    Try keyword-based retrieval first (good for very specific terms like 'pistachio').
    If that fails, fall back to embedding-based nearest-neighbour search.
    If that still fails, try a loose keyword fallback.
    """

    keyword_matches = find_keyword_matches(query, bc_documents, max_docs=20)
    if keyword_matches:
        return keyword_matches[:k]

    docs = retrieve_docs(query, k=k)
    if docs:
        return docs

    keywords = query.lower().split()
    alt_docs = []
    for kw in keywords:
        alt_docs.extend(retrieve_docs(kw, k=1))

    seen = set()
    unique = []
    for d in alt_docs:
        key = d.get("url") or d.get("title")
        if key and key not in seen:
            seen.add(key)
            unique.append(d)

    return unique[:k]


In [26]:
def summarize_query_with_rag(query, k=3):
    retrieved = safe_retrieve(query, k=k)

    if not retrieved:
        return f"No articles match your query: “{query}”"

    return fused_summary(retrieved, n_sentences=4)


In [27]:
print(summarize_query_with_rag("coastal first nations and the tanker ban", k=3))


The Canadian government is delaying its plan to introduce a bill to protect First Nations' right to clean, safe and affordable drinking water. The delay comes after a series of protests by First Nation communities against the planned expansion of the Port of Churchill, which would allow for more coal exports from Alberta.

This is not an isolated incident. In fact, this is just the latest example of how the Harper government has failed to live up to its promises to First Nations.

Sources:
- CBC Canada — Delayed introduction of First Nations clean water bill 'unacceptable,' say chiefs (https://www.cbc.ca/news/indigenous/first-nations-water-bill-9.7001699?cmp=rss)
- BBC World — Climate protesters in kayaks disrupt operations at Australian port (https://www.bbc.com/news/videos/c865p45j3qeo?at_medium=RSS&at_campaign=rss)
- BBC World — Cyclone catastrophe in Sri Lanka awakens volunteer spirit (https://www.bbc.com/news/articles/c058n10727po?at_medium=RSS&at_campaign=rss)


In [28]:
def rewrite_with_gpt2(summary, max_new_tokens=40):
    """
    Use GPT-2 to lightly rewrite the extractive summary for readability.
    We prepend strong instructions to avoid adding new facts, but GPT-2 can still hallucinate.
    """
    prompt = (
        "Rewrite the following news summary in clear, fluent English. "
        "Do NOT add any new information or details that are not already stated.\n\n"
        f"SUMMARY:\n{summary}\n\nREWRITE:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)
    output_ids = gpt2.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    rewritten = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return rewritten


In [123]:
def answer_news_query(user_query: str, k: int = 3):
    """
    1) Retrieve top-k docs (semantic search).
    2) Filter them using strong tokens from the query.
    3) If that fails, do a title-based keyword search over all docs.
    4) Fuse remaining docs into one extractive summary.
    """
    docs = safe_retrieve(user_query, k=k)
    if not docs:
        return "No relevant news found for this query."

    strong_tokens = get_strong_tokens(user_query)

    filtered = filter_docs_by_tokens(docs, strong_tokens, min_matches=2)

    if filtered:
        docs_to_summarize = filtered
    else:
        title_docs = title_keyword_search(strong_tokens, max_docs=k)
        if title_docs:
            docs_to_summarize = title_docs
        else:
            docs_to_summarize = docs[:1]

    final = fused_summary(docs_to_summarize, n_sentences=3)
    return final


In [67]:
def show_sample_headlines(n=15):
    print(f"Total articles loaded: {len(bc_documents)}\n")
    for i, d in enumerate(bc_documents[:n]):
        print(f"[{i}] {d['source']} — {d['title']}")
        print(f"    Link: {d.get('url', '')}")
        print()


In [66]:
show_sample_headlines(10)


Total articles loaded: 32

[0] Global News BC — B.C. pulp mill to close, leaving 350 employees out of work
    Link: https://globalnews.ca/news/11557907/bc-pulp-mill-crofton-close-forestry/

[1] Global News BC — John Rustad removed as B.C. Conservatives’ leader, party says
    Link: https://globalnews.ca/news/11557931/bc-conservative-john-rustad-not-resign/

[2] Global News BC — Husband of missing B.C. mother of 3 says RCMP are investigating him
    Link: https://globalnews.ca/news/11557808/husband-missing-bc-mother-rcmp-investigating-him/

[3] Global News BC — B.C. Supreme Court certifies class action lawsuit that Lytton wildfire was caused by train
    Link: https://globalnews.ca/news/11556625/bc-supreme-court-class-action-lawsuit-lytton-wildfire-caused-train/

[4] Global News BC — Friends, family of murdered B.C. mom gather in courtroom to read victim impact statements
    Link: https://globalnews.ca/news/11556396/friends-family-murdered-bc-mom-victim-impact-chelsey-gauthier/

[5] G

In [124]:
print("=== Crofton test ===")
print(answer_news_query("Summarize the story about the Crofton pulp mill closure", k=3))

print("\n=== Pistachio test ===")
print(answer_news_query("What happened with the pistachio salmonella restrictions?", k=3))

print("\n=== John Rustad test ===")
print(answer_news_query("Who is John Rustad and what issue is he involved in?", k=3))

print("\n=== Lytton wildfire lawsuit test ===")
print(answer_news_query("What did the court rule about the Lytton wildfire train case?", k=3))



=== Crofton test ===
Domtar, which owns the mill in Crofton, says continued poor pricing for pulp and a lack of access to affordable fibre led to the plant closure. The company said the pulp operations at the mill in Crofton, about 70 kilometres north of Victoria, have been struggling for a while.

Sources:
- Global News BC — B.C. pulp mill to close, leaving 350 employees out of work (https://globalnews.ca/news/11557907/bc-pulp-mill-crofton-close-forestry/)
- CBC BC — Domtar announces permanent closure of mill in Crofton, B.C. (https://www.cbc.ca/news/canada/british-columbia/domtar-pulp-mill-closure-crofton-9.7000784?cmp=rss)

=== Pistachio test ===
The outbreak of salmonella linked to pistachio and pistachio-containing products continues with more than 150 confirmed illnesses now reported. Iranian pistachios previously imported to Canada need to be held and tested for salmonella before being sold, federal officials say.

Sources:
- Global News BC — Canada puts new restrictions on pist